# Chess Data Analysis Pipeline

Tento notebook replikuje logiku ze souboru `main.py` a `data_analysis.py` a rozděluje ji do samostatných bloků pro snadnější spouštění a testování jednotlivých částí procesu analýzy šachových her.

## Konfigurace a příprava adresářů

In [ ]:
import os
import json
import os
from scipy.stats import pearsonr
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
import game_load
import stockfish_eval
import data_analysis

load_dotenv()

In [ ]:
CHESS_USER = os.getenv("CHESS_USER")
stockfish_path = "stockfish/stockfish-windows-x86-64-avx2.exe"

if CHESS_USER:
    path = CHESS_USER + "_data"
    if not os.path.isdir(path):
        os.mkdir(path)
        print(f"Vytvořena datová složka uživatele: {path}")
    print(f"Pracuji s uživatelem: {CHESS_USER}")
else:
    print("CHESS_USER není nastaven v .env souboru.")

## 1. Načtení her (Download Games)

In [ ]:
def load_games(base_path):
    games_dir = os.path.join(base_path, "games")
    if not os.path.isdir(games_dir):
        os.mkdir(games_dir)

    games_count = game_load.fetch_chess_games(CHESS_USER, games_dir)
    print(f"Staženo {games_count} her uživatele {CHESS_USER}")
    return games_dir

games_path = os.getenv("GAMES_PATH")

if not games_path or not os.path.isdir(games_path):
    print("Games path not found in env or dir missing. Loading games...")
    games_path = load_games(path)
else:
    print(f"Hry již nalezeny v: {games_path}")

## 2. Analýza her (Stockfish Evaluation)

In [ ]:
def analyze_games(games_source, base_path, engine_path):
    analyzed_dir = os.path.join(base_path, "games_analyzed")
    if not os.path.isdir(analyzed_dir):
        os.mkdir(analyzed_dir)

    print("Spouštím Stockfish analýzu...")
    count = stockfish_eval.analyze_archives(games_source, analyzed_dir, engine_path)
    print(f"Her analyzováno: {count}.")
    return analyzed_dir

analyzed_path = os.getenv("ANALYZED_PATH")

# Pokud není cesta v ENV, nebo složka neexistuje, je možné, že chceme spustit analýzu.
# Zde je to zakomentované, aby se nespustilo omylem (trvá dlouho).
# Odkomentujte pro spuštění:

# if not analyzed_path or not os.path.isdir(analyzed_path):
#     analyzed_path = analyze_games(games_path, path, stockfish_path)
# else:
#     print(f"Analyzovaná data nalezená v: {analyzed_path}")

# Pro účely notebooku, pokud cesta není definovaná, zkusíme default:
if not analyzed_path:
    analyzed_path = os.path.join(path, "games_analyzed")

print(f"Cesta k analyzovaným datům: {analyzed_path}")

## 3. Analýza Dat (Data Analysis & Statistics)

In [ ]:
df_games, df_moves = pd.DataFrame(), pd.DataFrame()
if not os.path.isdir(analyzed_path):
    print(f"Adresář {analyzed_path} neexistuje. Nejdříve spusťte analýzu her.")
elif len(os.listdir(analyzed_path)) == 0:
    print(f"Adresář {analyzed_path} je prázdný. Nejdříve spusťte analýzu her.")
else:
    for file in tqdm(reversed(os.listdir(analyzed_path)), total=len(os.listdir(analyzed_path)),
                     desc="\033[92mNačítání dat k analýze"):
        path = os.path.join(analyzed_path, file)
        df_games_file, df_moves_file = data_analysis.load_chess_data(path)
        df_games = pd.concat([df_games, df_games_file], ignore_index=True)
        df_moves = pd.concat([df_moves, df_moves_file], ignore_index=True)

    print(f"Celkem her načteno: {len(df_games)}")
    print(f"Celkem tahů načteno: {len(df_moves)}")

Niní máme data načtena v `df_games` a `df_moves`. `df_games` obsahuje informace o jednotlivých hrách a `df_moves` obsahuje podrobnosti o jednotlivých tazích extrahovaných ze všech her s dodatečným atributem `games_url` odkazující na původní hru z `df_games`, ve které byl tah zahrán.

### 3.1 Výpočet korelací (Correlation Calculations)